# External workflow logs — design loop

Tutorial notebook for **Scenario A**: record an external verification run on an Istari Digital system, react to a failure, re-run after a design change, and attach the passing report to the model.

You will:

1. Get or create a system with a tracked bracket model
2. Run a local verification battery (outside the platform)
3. Upload outputs and log a `FAILED` entry
4. Revise the design, re-run, log `SUCCESS`, and attach the verified report

Companion to [Scenario B — tradespace sweep](workflow_log_scenario_b.ipynb).

> **External workflows vs jobs.** External workflows run on your machine or in CI; Istari Digital stores the outputs and log entry. [Istari jobs](https://docs.istaridigital.com/developers/SDK/api_reference/03-jobs) run on platform agents and are scheduled end-to-end by the platform.

### Prerequisites

From the cookbook root:

```bash
uv sync --group dev --group advanced
```

| Group | Packages | Used for |
|---|---|---|
| **`dev`** | `istari-digital-client`, `python-dotenv` | Connect, systems, workflow log API |
| **`advanced`** | `pandas`, `jinja2`, `matplotlib`, `numpy`, `pytest`, `ipython` | Result tables, plots, tradespace sweep |

Other recipes need only `uv sync --group dev`.

- **Registry Service > 10.17.3** (2026-05 release or later)
- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
- **Experimental features** (Istari Digital web app → **Application Settings** → **Experimental Features**):
  - **Branching** — required for `commit_changes()` (snapshot + baseline tag) in §2, §7, and §9
  - **Workflow Log** — required for workflow outputs and log entries
- **`istari-digital-client` must be API-compatible with the registry** — Connect runs `Client.check_compatibility()` (SDK and registry version strings differ)
- Kernel: **Python (istari-client-cookbook)** (or any kernel backed by the synced `.venv`)

### Running order

Run cells top to bottom. Pause at the callout in §6 to review entries in the **Workflow log** tab.


## Scenario

A bracket (`.step` file) is tracked on a system. A verification battery runs ten checks. The initial design fails stress (fillet too small). After enlarging the fillet, the second run passes and the JUnit report is attached to the model.

Each run becomes a **workflow log entry** linked to the active **configuration**.


## 1. Connect

Load credentials from [`samples/.env`](../.env). Two clients share one `Configuration`:

- `Client` — read the system, download tracked files, update models
- `V3Client` — workflow outputs and workflow log entries

Connect checks registry release level and SDK API compatibility via `check_compatibility()`.

Requires Registry Service **> 10.17.3** (2026-05+). Connect reads `X-Istari-Registry-Version` and fails fast otherwise.


In [ ]:
import os, json, re
from importlib.metadata import version as pkg_version
from pathlib import Path
from dotenv import load_dotenv

from istari_helpers import MINIMUM_REGISTRY_VERSION, registry_meets_minimum

from istari_digital_client import Client, V3Client, Configuration
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]
PAT = os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"]

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", REGISTRY_URL)
UI_URL = REGISTRY_URL.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"

config = Configuration(registry_url=REGISTRY_URL, registry_auth_token=PAT)
client = Client(config)     # read system + download files
v3     = V3Client(config)   # workflow outputs + workflow log

installed = pkg_version("istari-digital-client")

compat = client.check_compatibility()
assert compat and compat.server_version, (
    "Registry did not return compatibility headers. "
    "Check ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env."
)
registry_version = compat.server_version

assert registry_meets_minimum(registry_version), (
    f"Istari Registry v{registry_version} does not meet the minimum for this recipe "
    f"(> {MINIMUM_REGISTRY_VERSION}, 2026-05 release). Point ISTARI_REGISTRY_URL at a "
    "2026-05+ deployment and install a matching istari-digital-client."
)

print("Connected to:", REGISTRY_URL)
print("UI:", UI_URL)
print(f"istari-digital-client {installed}")
print(f"Istari Registry v{registry_version} — compatible")

## 2. Get or create the demo system

Set `SYSTEM_NAME`, look up an active system with that name, and create it (with a `baseline` configuration) only when missing. Scenario A also seeds the initial R3 bracket model on first create. Ends with `commit_changes()` so the baseline branch shows the tracked files.


In [ ]:
import bracket_step, istari_helpers
from istari_digital_client.v2.models.new_system import NewSystem
from istari_digital_client.v2.models.new_system_configuration import NewSystemConfiguration
from istari_digital_client.v2.models.new_tracked_file import NewTrackedFile
from istari_digital_client.v2.models.tracked_file_specifier_type import TrackedFileSpecifierType

SYSTEM_NAME = "Workflow Log Walkthrough"
CONFIG_NAME = "baseline"

work = Path("_notebook_run"); work.mkdir(exist_ok=True)
src = work / "bracket.step"
src.write_text(bracket_step.bad())          # initial design: R3 fillet

system = istari_helpers.find_system_by_name(client, SYSTEM_NAME)
if system is None:
    model = client.add_model(path=src, display_name="bracket.step",
                             description="Bracket — initial R3 design")
    system = client.create_system(NewSystem(
        name=SYSTEM_NAME,
        description="External workflow log demo",
    ))
    config_obj = client.create_configuration(
        system_id=system.id,
        new_system_configuration=NewSystemConfiguration(
            name=CONFIG_NAME,
            tracked_files=[NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LATEST,
                file_id=model.file.id,
            )],
        ),
    )
    MODEL_ID, FILE_ID = model.id, model.file.id
    print("Created system:", system.id)
else:
    config_obj = istari_helpers.find_configuration(client, system.id, CONFIG_NAME)
    if config_obj is None:
        raise RuntimeError(f"System {SYSTEM_NAME!r} exists but has no {CONFIG_NAME!r} configuration")
    tracked = client.list_tracked_files(configuration_id=config_obj.id).items
    if not tracked:
        raise RuntimeError(f"Configuration {CONFIG_NAME!r} has no tracked files")
    FILE_ID = tracked[0].file_id
    MODEL_ID = istari_helpers.model_id_for_file(client, FILE_ID)
    if MODEL_ID is None:
        raise RuntimeError(f"No model found for file {FILE_ID}")
    print("Using existing system:", system.id)

SYSTEM_ID, CONFIG_ID = system.id, config_obj.id
istari_helpers.commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("System:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 3. Download the tracked file

External workflows typically start by fetching the file bytes to process locally.


In [ ]:
file_obj = client.get_file(file_id=FILE_ID)
source_bytes = file_obj.revisions[-1].read_bytes()

fillet = bracket_step.parse_fillet_mm(source_bytes)
print(f"Downloaded {len(source_bytes)} bytes")
print(f"Design parameter — fillet radius: {fillet:.1f} mm")

## 4. Run the verification battery

Ten checks run locally and write artifact files plus a JUnit summary. Stress fails on the R3 fillet — peak stress exceeds the allowable.


In [ ]:
import pandas as pd
from pathlib import Path
from run_workflow import run_battery, write_junit

work = Path("_notebook_run")
src = work / "bracket.step"
if not src.exists():
    raise FileNotFoundError(f"{src.resolve()} missing — run §2 first")
source_bytes = src.read_bytes()

results = run_battery(source_bytes, work / "artifacts")

def results_frame(results):
    df = pd.DataFrame([{"check": r.name,
                        "result": "PASS" if r.passed else "FAIL",
                        "detail": r.summary} for r in results])
    return df.style.map(
        lambda v: "color:#15803d;font-weight:600" if v == "PASS"
        else ("color:#dc2626;font-weight:600" if v == "FAIL" else ""),
        subset=["result"])

results_frame(results)

Sample outputs — thermal map and the failing stress check:

In [ ]:
from IPython.display import Image, display

art = work / "artifacts"
display(Image(filename=str(art / "04_thermal_map.png")))
print(json.dumps(json.loads((art / "05_stress_analysis.json").read_text()), indent=2))

In [ ]:
overall = "SUCCESS" if all(r.passed for r in results) else "FAILED"
junit = write_junit(results, art / "00_verification_results.xml", src.name)
print("Overall verdict:", overall)

## 5. Upload workflow outputs

Register each result file with `create_workflow_output`. Outputs live on the system but are not tracked configuration files.


In [ ]:
upload_paths = [junit] + [r.artifact_path for r in results]
output_ids = []
for p in upload_paths:
    wo = v3.create_workflow_output(system_id=SYSTEM_ID, path=p)
    output_ids.append(wo.id)
    print(f"  uploaded {p.name}")
print(f"\n{len(output_ids)} workflow outputs uploaded")

## 6. Record the workflow log entry

`create_workflow_log_entry` ties title, `status`, `configuration_id`, and output IDs into one durable record.

> **Pause here.** Open the system in the Istari Digital web app → **Workflow log** tab. Open the failed entry and preview the attached outputs.


In [ ]:
entry = v3.create_workflow_log_entry(
    system_id=SYSTEM_ID,
    workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
        title="Verification battery — iter-1 (initial design)",
        status=overall,                # SUCCESS | FAILED | UNSPECIFIED
        configuration_id=CONFIG_ID,
        workflow_output_ids=output_ids,
    ),
)
print("Created entry:", entry.id, "->", entry.status)
print("Open the Workflow log tab:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 7. Revise the design

Enlarge the fillet to R5 and upload a new model revision via the SDK (drag-and-drop in the web app achieves the same). Then `commit_changes()` snapshots the configuration and moves the baseline tag.


In [ ]:
src.write_text(bracket_step.good())         # redesign: R5 fillet
client.update_model(model_id=MODEL_ID, path=src,
                    description="Bracket — R5 redesign", version_name="v2-R5-fillet")

# Capture a new snapshot and advance the baseline so the system shows the R5 revision.
istari_helpers.commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("Committed new revision with fillet:",
      f"{bracket_step.parse_fillet_mm(src.read_bytes()):.1f} mm")

## 8. Re-run against the new revision

Same workflow code; `get_file` returns the latest revision. All checks pass.


In [ ]:
source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
print("Now testing fillet:", f"{bracket_step.parse_fillet_mm(source_bytes):.1f} mm")

results = run_battery(source_bytes, work / "artifacts2")
results_frame(results)

In [ ]:
art2 = work / "artifacts2"
overall = "SUCCESS" if all(r.passed for r in results) else "FAILED"
junit2 = write_junit(results, art2 / "00_verification_results.xml", src.name)

upload_paths = [junit2] + [r.artifact_path for r in results]
output_ids = [v3.create_workflow_output(system_id=SYSTEM_ID, path=p).id for p in upload_paths]

entry2 = v3.create_workflow_log_entry(
    system_id=SYSTEM_ID,
    workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
        title="Verification battery — iter-2 (R5 fillet redesign)",
        status=overall,
        configuration_id=CONFIG_ID,
        workflow_output_ids=output_ids,
    ),
)
print("Created entry:", entry2.id, "->", entry2.status)

## 9. Attach verified results to the model

Upload the passing JUnit report with `V3Client.create_resource()` (`resource_type=artifact`), link it to the bracket model via a `produces` revision relationship — the V3 Resources equivalent of `Client.add_artifact()` — then `commit_changes()` so the artifact appears under the model on the baseline snapshot (workflow outputs stay outside snapshots; model-linked artifacts do not).


In [ ]:
import shutil
from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto
from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

verified = work / "verified-test-results-for-bracket.step.xml"
shutil.copyfile(junit2, verified)

artifact = v3.create_resource(
    path=verified,
    resource_type=ResourceTypeDto.ARTIFACT,
    display_name=verified.name,
    description=f"Verified results (entry {entry2.id})",
)

model_resource = v3.get_resource(resource_id=MODEL_ID)
produces_type = next(
    t for t in v3.list_revision_relationship_types().items if t.name == "produces"
)
v3.create_revision_relationship(
    new_revision_relationship_dto=NewRevisionRelationshipDto(
        relationship_type_id=produces_type.id,
        left_revision_id=model_resource.file_revision_id,
        right_revision_id=artifact.file_revision_id,
    ),
)

# Capture the new artifact in a snapshot and advance baseline (like §7 after the redesign).
istari_helpers.commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("Artifact resource:", artifact.resource_id)
print("Linked to model:", MODEL_ID, f"via {produces_type.name}")
print("Committed to baseline snapshot")

## 10. Recap

On one system you now have:

- Two workflow log entries (`FAILED`, then `SUCCESS`), each linked to a configuration
- All result files from both runs, previewable in the web app
- A verified test report artifact resource linked to the model (`produces` relationship), committed on the baseline snapshot

Fail → revise → pass → evidence on the record, captured from an external workflow.


Open the **Workflow log** tab to review both entries:

In [ ]:
print("Review both scenarios in the Workflow log tab:")
print(f"  {UI_URL}/systems/{SYSTEM_ID}")

### Learn more

- [External workflow logs](https://docs.istaridigital.com/developers/SDK/v3/03-workflow-logs) — SDK reference for `create_workflow_output`, `create_workflow_log_entry`, and listing entries
- **Workflow log** tab on a system (requires the experimental feature above)


## Teardown (optional)

Archive the demo system when you are finished so repeated runs do not clutter the instance. Safe to skip during a live walkthrough; archiving is reversible.


In [ ]:
client.archive_system(system_id=SYSTEM_ID)
print("Archived system", SYSTEM_ID)